We cannot give text to logistic or svm or similar models. 
So what do we do? How do we train our model on basis of text??
Answer to this is - we train them on term frequency and inverse document freqency 

Count Vectorizer - It counts number of times each unique word comes. 
And then it counts for each word for each document. For example - if call comes in only 1 message - and not in 3 messages - so it will have only in one doc. And this matrix is called document term matrix. (DTM) 

Other way we have is TF-IDF - term frequency - inverse document frequency vectorizer. 
It doesnt fill DTM with token counts, it calculates tf-idf for each word. 

TF - term frequency - number of times a word occurs in a document. TF of word 'xyz' in doc1 - 0, doc-2, doc3-1.

Though its just the raw count. Its not fine on its own,becvause we have stop words like the , a , and other usual words. If we take just Term frequency then we will be giving weightage to stop words more than we should give, and main words will be given less weightage. 

So we have inverse document frequency - it deminishes the regular words weightage. 

TF-IDF = term frequency * (1/ document frequency) 
TF-IDF = term frequency * inverse document frequency. 

idf(t,d) = log[ N/|{d ∈ D : t ∈ d}| ] 

For us SKlearn can calculate all these things. 

from sklearb.feature_extraction.text import TfidfVectorizer 

obj = TfidfVectorizer()
dtm = obj.fit_transform(message) 

- and this will make the dtm matrix. On which we can train our model 

In [2]:
import sklearn
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv("L:\\Workspace\\Resources\\UPDATED_NLP_COURSE\\TextFiles\\smsspamcollection.tsv", sep='\t')

In [5]:
df.head()

,label,message,length,punct
0,ham,"Go until jurong point, crazy.. Available only ...",111,9
1,ham,Ok lar... Joking wif u oni...,29,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,6
3,ham,U dun say so early hor... U c already then say...,49,6
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,2


In [6]:
df.isnull().sum()

label      0
message    0
length     0
punct      0
dtype: int64

In [8]:
df[df['label']=='']

,label,message,length,punct


In [9]:
df[df['message']=='']

,label,message,length,punct


In [10]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
X = df['message']
y = df['label']

In [14]:
x_train, x_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

In [16]:
x_train.head()

708     Quite late lar... Ard 12 anyway i wun b drivin...
4338                        on a Tuesday night r u 4 real
5029    Go chase after her and run her over while she'...
4921     G says you never answer your texts, confirm/deny
2592         Still work going on:)it is very small house.
Name: message, dtype: object

In [17]:
y_test.head()

3245    ham
944     ham
1044    ham
2484    ham
812     ham
Name: label, dtype: object

Now we do count vectorizing. We text pre processing, stop words, includes in count vectorization 

In [19]:
from sklearn.feature_extraction.text import CountVectorizer

In [20]:
cv = CountVectorizer()

1. Fit the vectorizer to the data - this builds vocab, count words and all.
2. Then we transform the original text message to vectors. 

In [21]:
# cv.fit(x_train)
# x_train_counts = cv.trasnform(x_train)

# or - 
x_train_counts = cv.fit_transform(x_train)

In [22]:
x_train_counts

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 52150 stored elements and shape (3900, 7263)>

In [23]:
x_train.shape

(3900,)

In [24]:
x_train_counts.shape

(3900, 7263)

We just created a count vectorization 

Now we will use tf idf 

We can give imp words more weightage using tf-idf 

In [25]:
from sklearn.feature_extraction.text import TfidfTransformer

In [26]:
tf = TfidfTransformer()

In [27]:
x_train_tfidf = tf.fit_transform(x_train_counts)

In [28]:
x_train_tfidf.shape

(3900, 7263)

Though we can skip 2 diff steps - count vectorization and then tf idf, 
We can do this - vectorizer + tfidf 

from sklearn.feature_extraction.text improt TfidfVectorizer 
vect = TfidfVectorizer()
x_train_tfidf = vect.fit_trainsform(x_train)

In [29]:
from sklearn.svm import LinearSVC

In [30]:
clf = LinearSVC()
clf.fit(x_train_tfidf, y_train)

LinearSVC()

Instead of writing all this - we can just use pipeline so that we dont have to do everything again and again . 
lets do it 

In [33]:
from sklearn.pipeline import Pipeline

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer 

In [35]:
text_clf = Pipeline([('tfidf',TfidfVectorizer()), ('clf',LinearSVC())])

In [36]:
# IK this is a short pipeline for now 

In [38]:
text_clf.fit(x_train, y_train) 

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', LinearSVC())])

In [39]:
pred = text_clf.predict(x_test)

In [40]:
pred

array(['ham', 'ham', 'ham', ..., 'ham', 'spam', 'ham'], dtype=object)

In [41]:
from sklearn.metrics import confusion_matrix, classification_report

In [43]:
print(confusion_matrix(y_test, pred))

[[1445    3]
 [  10  214]]


In [44]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

         ham       0.99      1.00      1.00      1448
        spam       0.99      0.96      0.97       224

    accuracy                           0.99      1672
   macro avg       0.99      0.98      0.98      1672
weighted avg       0.99      0.99      0.99      1672



In [45]:
from sklearn import metrics

In [46]:
metrics.accuracy_score(y_test, pred)

0.9922248803827751

In [47]:
text_clf.predict(['Hi how are you doing today?'])

array(['ham'], dtype=object)

In [49]:
text_clf.predict(['Congratulations! You have been selected as a winner for lottery'])

array(['spam'], dtype=object)